<div dir = "rtl">

### چرا Feature Engineering مهم است؟

در خیلی از پروژه‌های داده، کیفیت ویژگی‌ها حتی از انتخاب مدل مهم‌تر است.

اگر ویژگی‌های خوبی بسازیم، مدل‌های ساده هم می‌توانند عملکرد خوبی داشته باشند.

اما اگر ویژگی‌ها ضعیف باشند، حتی مدل‌های قوی هم خوب عمل نمی‌کنند.

در این پروژه، چون داده ما ماهیت زمانی دارد، باید ویژگی‌هایی بسازیم که تغییرات گذشته، روند، فصل، و نوسان را به مدل نشان دهند.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import sys
project_root = Path.cwd().parent
if str(project_root)not in sys.path:
    sys.path.append(str(project_root))
from src.features import (
    load_data,
    add_basic_time_features,
    add_season_features,
    add_error_features,
    add_lag_features,
    add_rolling_features,
    add_difference_features,
    build_feature_dataset,
    save_feature_dataset
)


In [2]:
DATA_DIR = project_root / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DB_PATH = project_root / "db" / "energy_load.db"

df = load_data(DB_PATH)
df.head()


,source_file,interval_start,interval_end,week_label,week_number,area_name,area_code,forecast_min_mw,actual_min_mw,actual_max_mw,forecast_max_mw,data_year,created_at
0,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 1,1,Poland,PL,12898.5,12992.96,24945.56,27700.0,2019,2026-07-06 20:09:35
1,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 2,2,Poland,PL,14170.9,15513.79,25954.28,27700.0,2019,2026-07-06 20:09:35
2,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 3,3,Poland,PL,15434.4,15798.93,25264.46,27700.0,2019,2026-07-06 20:09:35
3,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 4,4,Poland,PL,15483.2,16055.43,26135.58,27700.0,2019,2026-07-06 20:09:35
4,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 5,5,Poland,PL,15090.6,14985.68,25339.89,27700.0,2019,2026-07-06 20:09:35


<div dir = "rtl">

### بررسی اولیه

In [3]:
df.shape

(418, 13)

In [4]:
df.columns.tolist()

['source_file',
 'interval_start',
 'interval_end',
 'week_label',
 'week_number',
 'area_name',
 'area_code',
 'forecast_min_mw',
 'actual_min_mw',
 'actual_max_mw',
 'forecast_max_mw',
 'data_year',
 'created_at']

<div dir = "rtl">

### ساخت ویژگی‌های پایه زمانی


در ساده‌ترین حالت، دانستن اینکه داده مربوط به کدام سال و هفته است، به مدل کمک می‌کند.

مثلاً ممکن است هفته ۱ و هفته ۵۲ رفتار مشابهی داشته باشند یا بعضی فصل‌ها بار بیشتری داشته باشند.

ویژگی‌هایی که در این بخش ساختیم:

-    year
-    month
-    quarter
-    week_of_year

In [5]:
basic_df = add_basic_time_features(df)
basic_df[["interval_end", "year", "month", "quarter", "week_of_year"]].head()

,interval_end,year,month,quarter,week_of_year
0,2019-12-30,2019,12,4,1
1,2019-12-30,2019,12,4,2
2,2019-12-30,2019,12,4,3
3,2019-12-30,2019,12,4,4
4,2019-12-30,2019,12,4,5


<div dir = "rtl">

### ساخت ویژگی‌های فصلی

فصل‌ها روی مصرف برق اثر زیادی دارند.

مثلاً در زمستان یا تابستان ممکن است مصرف برق به‌خاطر گرمایش یا سرمایش بیشتر شود.

برای همین:

    فصل را از روی ماه ساختیم
    برای هر فصل indicator تعریف کردیم
    از week_sin و week_cos استفاده کردیم تا خاصیت چرخه‌ای هفته‌ها را حفظ کنیم

چرا sin و cos؟

چون هفته ۱ و هفته ۵۲ از نظر زمانی به هم نزدیک‌اند، اما اگر فقط عدد خام هفته را بدهیم، مدل فکر می‌کند فاصله آن‌ها خیلی زیاد است.

تبدیل سینوسی و کسینوسی این مشکل را حل می‌کند.

In [6]:
season_df = add_season_features(basic_df)
season_df[["week_number", "month", "season", "is_winter", "is_summer", "week_sin", "week_cos"]].head()


,week_number,month,season,is_winter,is_summer,week_sin,week_cos
0,1,12,winter,1,0,0.120537,0.992709
1,2,12,winter,1,0,0.239316,0.970942
2,3,12,winter,1,0,0.354605,0.935016
3,4,12,winter,1,0,0.464723,0.885456
4,5,12,winter,1,0,0.568065,0.822984


<div dir = "rtl">

### ساخت ویژگی‌های خطا

In [7]:
error_df = add_error_features(season_df)
error_df[["error_min_mw", "error_max_mw", "abs_error_min_mw", "abs_error_max_mw"]].head()


,error_min_mw,error_max_mw,abs_error_min_mw,abs_error_max_mw
0,94.46,-2754.44,94.46,2754.44
1,1342.89,-1745.72,1342.89,1745.72
2,364.53,-2435.54,364.53,2435.54
3,572.23,-1564.42,572.23,1564.42
4,-104.92,-2360.11,104.92,2360.11


<div dir = "rtl">

### ساخت lag features

 یعنی مقدارهای گذشته.

مثلاً:

-    actual_max_mw_lag_1 یعنی مقدار actual_max_mw در هفته قبل

-    actual_max_mw_lag_2 یعنی دو هفته قبل

این ویژگی‌ها بسیار مهم‌اند، چون در داده‌های زمانی، گذشته معمولاً بهترین سرنخ برای آینده است.

در این پروژه برای هر ستون اصلی این lagها را ساختیم:

-    1 هفته قبل
-    2 هفته قبل
-    3 هفته قبل
-    4 هفته قبل

In [8]:
lag_df = add_lag_features(error_df, lags=[1, 2, 3, 4])
lag_df.filter(regex="lag").head()


,actual_min_mw_lag_1,actual_min_mw_lag_2,actual_min_mw_lag_3,actual_min_mw_lag_4,actual_max_mw_lag_1,actual_max_mw_lag_2,actual_max_mw_lag_3,actual_max_mw_lag_4,forecast_min_mw_lag_1,forecast_min_mw_lag_2,forecast_min_mw_lag_3,forecast_min_mw_lag_4,forecast_max_mw_lag_1,forecast_max_mw_lag_2,forecast_max_mw_lag_3,forecast_max_mw_lag_4
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12992.96,NaN,NaN,NaN,24945.56,NaN,NaN,NaN,12898.5,NaN,NaN,NaN,27700.0,NaN,NaN,NaN
2,15513.79,12992.96,NaN,NaN,25954.28,24945.56,NaN,NaN,14170.9,12898.5,NaN,NaN,27700.0,27700.0,NaN,NaN
3,15798.93,15513.79,12992.96,NaN,25264.46,25954.28,24945.56,NaN,15434.4,14170.9,12898.5,NaN,27700.0,27700.0,27700.0,NaN
4,16055.43,15798.93,15513.79,12992.96,26135.58,25264.46,25954.28,24945.56,15483.2,15434.4,14170.9,12898.5,27700.0,27700.0,27700.0,27700.0


<div dir = "rtl">

###  ساخت rolling features

In [9]:
rolling_df = add_rolling_features(lag_df, windows=[3, 4, 8])
rolling_df.filter(regex="roll").head()


,actual_min_mw_roll_mean_3,actual_min_mw_roll_std_3,actual_min_mw_roll_mean_4,actual_min_mw_roll_std_4,actual_min_mw_roll_mean_8,actual_min_mw_roll_std_8,actual_max_mw_roll_mean_3,actual_max_mw_roll_std_3,actual_max_mw_roll_mean_4,actual_max_mw_roll_std_4,...,forecast_min_mw_roll_mean_4,forecast_min_mw_roll_std_4,forecast_min_mw_roll_mean_8,forecast_min_mw_roll_std_8,forecast_max_mw_roll_mean_3,forecast_max_mw_roll_std_3,forecast_max_mw_roll_mean_4,forecast_max_mw_roll_std_4,forecast_max_mw_roll_mean_8,forecast_max_mw_roll_std_8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,14768.560000,1544.309789,NaN,NaN,NaN,NaN,25388.100000,515.600763,NaN,NaN,...,NaN,NaN,NaN,NaN,27700.0,0.0,NaN,NaN,NaN,NaN
4,15789.383333,270.946169,15090.2775,1415.604847,NaN,NaN,25784.773333,459.632349,25574.97,562.948505,...,14496.75,1226.492616,NaN,NaN,27700.0,0.0,27700.0,0.0,NaN,NaN


<div dir = "rtl">

### ساخت difference features

In [10]:
diff_df = add_difference_features(rolling_df)
diff_df.filter(regex="diff|pct_change|range").head()

,actual_min_mw_diff_1,actual_min_mw_pct_change_1,actual_max_mw_diff_1,actual_max_mw_pct_change_1,forecast_min_mw_diff_1,forecast_min_mw_pct_change_1,forecast_max_mw_diff_1,forecast_max_mw_pct_change_1,range_actual_mw,range_forecast_mw,range_actual_diff_1,range_forecast_diff_1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11952.60,14801.5,NaN,NaN
1,2520.83,0.194015,1008.72,0.040437,1272.4,0.098647,0.0,0.0,10440.49,13529.1,-1512.11,-1272.4
2,285.14,0.018380,-689.82,-0.026578,1263.5,0.089162,0.0,0.0,9465.53,12265.6,-974.96,-1263.5
3,256.50,0.016235,871.12,0.034480,48.8,0.003162,0.0,0.0,10080.15,12216.8,614.62,-48.8
4,-1069.75,-0.066629,-795.69,-0.030445,-392.6,-0.025357,0.0,0.0,10354.21,12609.4,274.06,392.6


<div dir = "rtl">

### ساخت دیتاست نهایی ویژگی‌ها

In [11]:
feature_df = build_feature_dataset(df)
feature_df.head()


,source_file,interval_start,interval_end,week_label,week_number,area_name,area_code,forecast_min_mw,actual_min_mw,actual_max_mw,...,actual_max_mw_diff_1,actual_max_mw_pct_change_1,forecast_min_mw_diff_1,forecast_min_mw_pct_change_1,forecast_max_mw_diff_1,forecast_max_mw_pct_change_1,range_actual_mw,range_forecast_mw,range_actual_diff_1,range_forecast_diff_1
0,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 1,1,Poland,PL,12898.5,12992.96,24945.56,...,NaN,NaN,NaN,NaN,NaN,NaN,11952.60,14801.5,NaN,NaN
1,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 2,2,Poland,PL,14170.9,15513.79,25954.28,...,1008.72,0.040437,1272.4,0.098647,0.0,0.0,10440.49,13529.1,-1512.11,-1272.4
2,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 3,3,Poland,PL,15434.4,15798.93,25264.46,...,-689.82,-0.026578,1263.5,0.089162,0.0,0.0,9465.53,12265.6,-974.96,-1263.5
3,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 4,4,Poland,PL,15483.2,16055.43,26135.58,...,871.12,0.034480,48.8,0.003162,0.0,0.0,10080.15,12216.8,614.62,-48.8
4,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31,2019-12-30,Week 5,5,Poland,PL,15090.6,14985.68,25339.89,...,-795.69,-0.030445,-392.6,-0.025357,0.0,0.0,10354.21,12609.4,274.06,392.6


<div dir = "rtl">

### بررسی ستون‌های تولیدشده

In [12]:
feature_df.columns.tolist()


['source_file',
 'interval_start',
 'interval_end',
 'week_label',
 'week_number',
 'area_name',
 'area_code',
 'forecast_min_mw',
 'actual_min_mw',
 'actual_max_mw',
 'forecast_max_mw',
 'data_year',
 'created_at',
 'year',
 'month',
 'quarter',
 'week_of_year',
 'season',
 'is_winter',
 'is_spring',
 'is_summer',
 'is_autumn',
 'week_sin',
 'week_cos',
 'error_min_mw',
 'error_max_mw',
 'abs_error_min_mw',
 'abs_error_max_mw',
 'actual_min_mw_lag_1',
 'actual_min_mw_lag_2',
 'actual_min_mw_lag_3',
 'actual_min_mw_lag_4',
 'actual_max_mw_lag_1',
 'actual_max_mw_lag_2',
 'actual_max_mw_lag_3',
 'actual_max_mw_lag_4',
 'forecast_min_mw_lag_1',
 'forecast_min_mw_lag_2',
 'forecast_min_mw_lag_3',
 'forecast_min_mw_lag_4',
 'forecast_max_mw_lag_1',
 'forecast_max_mw_lag_2',
 'forecast_max_mw_lag_3',
 'forecast_max_mw_lag_4',
 'actual_min_mw_roll_mean_3',
 'actual_min_mw_roll_std_3',
 'actual_min_mw_roll_mean_4',
 'actual_min_mw_roll_std_4',
 'actual_min_mw_roll_mean_8',
 'actual_min_mw_rol

<div dir = "rtl">

### بررسی مقادیر گمشده پس از feature engineering

In [13]:
feature_df.isna().sum().sort_values(ascending=False).head(30)


actual_min_mw_roll_mean_8     33
actual_min_mw_roll_std_8      33
actual_max_mw_roll_mean_8     33
actual_max_mw_roll_std_8      33
actual_max_mw_roll_std_4      29
actual_max_mw_roll_mean_4     29
actual_min_mw_roll_std_4      29
actual_min_mw_roll_mean_4     29
actual_min_mw_roll_std_3      28
actual_min_mw_roll_mean_3     28
actual_max_mw_roll_std_3      28
actual_max_mw_roll_mean_3     28
actual_min_mw_diff_1          27
actual_min_mw_pct_change_1    27
actual_max_mw_diff_1          27
range_actual_diff_1           27
actual_max_mw_pct_change_1    27
actual_max_mw_lag_4           26
range_actual_mw               26
actual_max_mw_lag_3           26
actual_min_mw_lag_4           26
actual_min_mw_lag_3           26
actual_min_mw_lag_2           26
actual_min_mw_lag_1           26
abs_error_max_mw              26
abs_error_min_mw              26
actual_max_mw_lag_2           26
actual_max_mw_lag_1           26
error_max_mw                  26
error_min_mw                  26
dtype: int

<div dir = "rtl">

### دلیل وجود NaN در lag و rolling

In [14]:
feature_df[[
    "area_code", "interval_end",
    "actual_max_mw",
    "actual_max_mw_lag_1",
    "actual_max_mw_lag_2",
    "actual_max_mw_roll_mean_3",
    "actual_max_mw_roll_std_3"
]].head(10)


,area_code,interval_end,actual_max_mw,actual_max_mw_lag_1,actual_max_mw_lag_2,actual_max_mw_roll_mean_3,actual_max_mw_roll_std_3
0,PL,2019-12-30,24945.56,NaN,NaN,NaN,NaN
1,PL,2019-12-30,25954.28,24945.56,NaN,NaN,NaN
2,PL,2019-12-30,25264.46,25954.28,24945.56,NaN,NaN
3,PL,2019-12-30,26135.58,25264.46,25954.28,25388.100000,515.600763
4,PL,2019-12-30,25339.89,26135.58,25264.46,25784.773333,459.632349
5,PL,2019-12-30,24931.56,25339.89,26135.58,25579.976667,482.642434
6,PL,2019-12-30,24699.60,24931.56,25339.89,25469.010000,612.307130
7,PL,2019-12-30,24560.56,24699.60,24931.56,24990.350000,324.168196
8,PL,2019-12-30,24220.56,24560.56,24699.60,24730.573333,187.429348
9,PL,2019-12-30,23769.44,24220.56,24560.56,24493.573333,246.445208


<div dir = "rtl">

### ذخیره خروجی نهایی

In [15]:
csv_path, db_path, table_name = save_feature_dataset(feature_df)
csv_path, db_path, table_name


(WindowsPath('d:/my computer/taha/zohram/Analyture/02_project/01_energy-load-project/data/processed/weekly_load_features.csv'),
 WindowsPath('d:/my computer/taha/zohram/Analyture/02_project/01_energy-load-project/db/energy_load.db'),
 'weekly_load_features')

<div dir = "rtl">

### بررسی دیتاست ذخیره‌شده در SQLite

In [16]:
conn = sqlite3.connect(DB_PATH)
preview = pd.read_sql_query("SELECT * FROM weekly_load_features LIMIT 10", conn)
conn.close()

preview.head()


,source_file,interval_start,interval_end,week_label,week_number,area_name,area_code,forecast_min_mw,actual_min_mw,actual_max_mw,...,actual_max_mw_diff_1,actual_max_mw_pct_change_1,forecast_min_mw_diff_1,forecast_min_mw_pct_change_1,forecast_max_mw_diff_1,forecast_max_mw_pct_change_1,range_actual_mw,range_forecast_mw,range_actual_diff_1,range_forecast_diff_1
0,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31 00:00:00,2019-12-30 00:00:00,Week 1,1,Poland,PL,12898.5,12992.96,24945.56,...,NaN,NaN,NaN,NaN,NaN,NaN,11952.60,14801.5,NaN,NaN
1,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31 00:00:00,2019-12-30 00:00:00,Week 2,2,Poland,PL,14170.9,15513.79,25954.28,...,1008.72,0.040437,1272.4,0.098647,0.0,0.0,10440.49,13529.1,-1512.11,-1272.4
2,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31 00:00:00,2019-12-30 00:00:00,Week 3,3,Poland,PL,15434.4,15798.93,25264.46,...,-689.82,-0.026578,1263.5,0.089162,0.0,0.0,9465.53,12265.6,-974.96,-1263.5
3,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31 00:00:00,2019-12-30 00:00:00,Week 4,4,Poland,PL,15483.2,16055.43,26135.58,...,871.12,0.034480,48.8,0.003162,0.0,0.0,10080.15,12216.8,614.62,-48.8
4,GUI_TOTAL_LOAD_YEARAHEAD_201812312300-20191231...,2018-12-31 00:00:00,2019-12-30 00:00:00,Week 5,5,Poland,PL,15090.6,14985.68,25339.89,...,-795.69,-0.030445,-392.6,-0.025357,0.0,0.0,10354.21,12609.4,274.06,392.6
